In [1]:
import sympy as sym
from sympy import *
import numpy as np
from tabulate import tabulate

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
#constants for 11B nuclei
Ispin = 3/2
w0 = 192.55 #Larmor Frequency for 11B (MHz)
wkhz = w0*10**3 # LArmor frequency in Hz

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*(wkhz))

# Coefficient for RHQ (cluster 2) from ASICS (in kHz)
A_coeff = [-2.767572, -2.045625, -2.035060]
B_coeff = [-0.021837, 0.688740, -0.697897]
C_coeff = [-2.611411, -1.907098, 0.778273]
D_coeff = [0.910820, -0.481464, -0.288617]
E_coeff = [0.405120, 0.489621, 0.805603]


In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [6]:
#Define symbol for quadrupolar tensor and force them to be real
AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q = sym.symbols('AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q', real=True)

#Variable for each equation set
quad_tensor = [(AzzmAyy_Q, Ayz_Q), (AzzmAxx_Q, Axz_Q), (AyymAxx_Q, Axy_Q)]

# List to hold solutions for quadrupolar tensor terms
solutions_Q = []

for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
    
    eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q/8)), D_coeff[i])
    eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q/2), E_coeff[i])

    # Solve the system
    solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
    solutions_Q.append(solution)

# Assign the solutions to the respective variables
AzzmAyy_Q = [solutions_Q[0][0][0], solutions_Q[0][1][0]]
Ayz_Q = [solutions_Q[0][0][1], solutions_Q[0][1][1]]
AzzmAxx_Q = [solutions_Q[1][0][0], solutions_Q[1][1][0]]
Axz_Q = [solutions_Q[1][0][1], solutions_Q[1][1][1]]
AyymAxx_Q = [solutions_Q[2][0][0], solutions_Q[2][1][0]]
Axy_Q = [solutions_Q[2][0][1], solutions_Q[2][1][1]]

# print(solutions_Q)
# Print the final results for the variables
print(f"Azz - Ayy: {AzzmAyy_Q}, Ayz: {Ayz_Q}")
print(f"Azz - Axx: {AzzmAxx_Q}, Axz: {Axz_Q}")
print(f"Ayy - Axx: {AyymAxx_Q}, Axy: {Axy_Q}")  



Azz - Ayy: [-466.553876261211, 466.553876261211], Ayz: [49.5395010056286, -49.5395010056286]
Azz - Axx: [-153.024434811385, 153.024434811385], Axz: [182.544603350348, -182.544603350348]
Ayy - Axx: [-254.383901514265, 254.383901514265], Axy: [180.676303546787, -180.676303546787]


In [7]:
import itertools
# Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
combinations = list(itertools.product(AzzmAxx_Q, AyymAxx_Q, AzzmAyy_Q))
print('Combination of (Azz - Axx), (Ayy - Axx), (Azz - Ayy): \n ', combinations)

Combination of (Azz - Axx), (Ayy - Axx), (Azz - Ayy): 
  [(-153.024434811385, -254.383901514265, -466.553876261211), (-153.024434811385, -254.383901514265, 466.553876261211), (-153.024434811385, 254.383901514265, -466.553876261211), (-153.024434811385, 254.383901514265, 466.553876261211), (153.024434811385, -254.383901514265, -466.553876261211), (153.024434811385, -254.383901514265, 466.553876261211), (153.024434811385, 254.383901514265, -466.553876261211), (153.024434811385, 254.383901514265, 466.553876261211)]


In [8]:

#Find Quadrupolar tensor diagonal elements

Axx1 = []; Axx2 = []; Axx3 = []
Ayy1 = []; Ayy2 = []; Ayy3 = []
Azz1 = []; Azz2 = []; Azz3 = []

# Initialize variables to track the best combination and minimum variation
best_combination = None
min_variation = float('inf')
for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
    # Solution 1
    Axx1_val = (-(AzzmAxx_val + AyymAxx_val)/3)
    Ayy1_val = Axx1_val + AyymAxx_val
    Azz1_val = Axx1_val + AzzmAxx_val

    #Save values
    Axx1.append(Axx1_val)
    Ayy1.append(Ayy1_val)
    Azz1.append(Azz1_val)

     # Solution 2
    Ayy2_val = -(AzzmAyy_val - AyymAxx_val) / 3
    Axx2_val = Ayy2_val - AyymAxx_val
    Azz2_val = Ayy2_val + AzzmAyy_val

     #Save values
    Axx2.append(Axx2_val)
    Ayy2.append(Ayy2_val)
    Azz2.append(Azz2_val)

    # Solution 3
    Azz3_val = (AzzmAxx_val + AzzmAyy_val) / 3
    Axx3_val = Azz3_val - AzzmAxx_val
    Ayy3_val = Azz3_val - AzzmAyy_val
    
    #Save values
    Axx3.append(Axx3_val)
    Ayy3.append(Ayy3_val)
    Azz3.append(Azz3_val)

    # Convert sympy Float to regular Python float for NumPy functions
    Axx1_val = float(Axx1_val)
    Axx2_val = float(Axx2_val)
    Axx3_val = float(Axx3_val)
    
    Ayy1_val = float(Ayy1_val)
    Ayy2_val = float(Ayy2_val)
    Ayy3_val = float(Ayy3_val)
    
    Azz1_val = float(Azz1_val)
    Azz2_val = float(Azz2_val)
    Azz3_val = float(Azz3_val)

    # Calculate variation (standard deviation) for Axx, Ayy, Azz
    variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
    variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
    variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

    total_variation = variation_Axx + variation_Ayy + variation_Azz

    # Update the best combination if the current one has less variation
    if total_variation < min_variation:
        min_variation = total_variation
        best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
        best_Axx_Q = np.mean([Axx1_val, Axx2_val, Axx3_val])
        best_Ayy_Q = np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
        best_Azz_Q = np.mean([Azz1_val, Azz2_val, Azz3_val])

#Get index for off-diagonal elements        
index_AzzmAxx = AzzmAxx_Q.index(best_combination[0])
best_Axz_Q = Axz_Q[index_AzzmAxx]

index_AyymAxx = AyymAxx_Q.index(best_combination[1])
best_Axy_Q = Axy_Q[index_AyymAxx]

index_AzzmAyy = AzzmAyy_Q.index(best_combination[2])
best_Ayz_Q = Ayz_Q[index_AzzmAyy]

# Print results
print("Axx1:", Axx1)
print("Axx2:", Axx2)
print("Axx3:", Axx3)

print("Ayy1:", Ayy1)
print("Ayy2:", Ayy2)
print("Ayy3:", Ayy3)

print("Azz1:", Azz1)
print("Azz2:", Azz2)
print("Azz3:", Azz3)

print("Best combination with minimum standard deviation:")
print("AzzmAxx:", best_combination[0])
print("AyymAxx:", best_combination[1])
print("AzzmAyy:", best_combination[2])


print("Axz:", best_Axz_Q)
print("Axy:", best_Axy_Q)
print("Ayz:", best_Ayz_Q)

print("Average of Axx1, Axx2, Axx3 with minimum standard deviation:", best_Axx_Q)
print("Average of Ayy1, Ayy2, Ayy3 with minimum standard deviation:", best_Ayy_Q)
print("Average of Azz1, Azz2, Azz3 with minimum standard deviation:", best_Azz_Q)




Axx1: [135.802778775217, 135.802778775217, -33.7864889009600, -33.7864889009600, 33.7864889009600, 33.7864889009600, -135.802778775217, -135.802778775217]
Axx2: [325.107226429914, 14.0713089224398, -14.0713089224398, -325.107226429914, 325.107226429914, 14.0713089224398, -14.0713089224398, -325.107226429914]
Axx3: [-53.5016688794801, 257.534248627994, -53.5016688794801, 257.534248627994, -257.534248627994, 53.5016688794801, -257.534248627994, 53.5016688794801]
Ayy1: [-118.581122739048, -118.581122739048, 220.597412613305, 220.597412613305, -220.597412613305, -220.597412613305, 118.581122739048, 118.581122739048]
Ayy2: [70.7233249156486, -240.312592591825, 240.312592591825, -70.7233249156486, 70.7233249156486, -240.312592591825, 240.312592591825, -70.7233249156486]
Ayy3: [260.027772570346, -362.044062444602, 260.027772570346, -362.044062444602, 362.044062444602, -260.027772570346, 362.044062444602, -260.027772570346]
Azz1: [-17.2216560361684, -17.2216560361684, -186.810923712345, -186.8

In [9]:
#Define symbol for CSA tensor and force them to be real
Azz_s, Axx_s, Ayy_s, Ayz_s, Axz_s, Axy_s = sym.symbols('Azz_s,Axx_s,Ayy_s,Ayz_s,Axz_s,Axy_s', real=True)

#Variables for each equation
cs_tensor = [(Ayy_s, Azz_s, Ayz_s), # for x -> Abb = Ayy; Agg = Azz; Abg = Ayz
             (Axx_s, Azz_s, Axz_s), # for y -> Abb = Axx; Agg = Azz; Abg = Axz
             (Axx_s, Ayy_s, Axy_s)] # for z -> Abb = Axx; Agg = Ayy; Abg = Axy

#Store variables in dictionary for access
A = {
    'xx': best_Axx_Q, 'yy': best_Ayy_Q, 'zz': best_Azz_Q,
    'yz': best_Ayz_Q, 'zy': best_Ayz_Q,
    'xz': best_Axz_Q, 'zx': best_Axz_Q,
    'xy': best_Axy_Q, 'yx': best_Axy_Q,
}

#Define rotation tuple (a, b, g, bg, m)
rotations = [
    ('x', 'y', 'z', 'yz', 1),   # a = x, b = y, g = z, m = 1
    ('y', 'x', 'z', 'xz', 1),  # a = y, b = x, g = z, m = 1
    ('z', 'x', 'y', 'xy', -1)  # a = z, b = x, g = y, m = -1
]

# List to hold solutions
solutions_cs = []

for i, (Abb_s, Agg_s, Abg_s) in enumerate(cs_tensor):
    a, b, g, bg, m = rotations[i]
    eq1 = sym.Eq(
        (8*A[a+a]*(A[b+b] + A[g+g] - A[a+a]) + 16*(A[a+b]**2 + A[a+g]**2) + 5*(A[b+b]**2 + A[g+g]**2) + 28*A[b+g]**2 - 18*A[b+b]*A[g+g])*(q/8) - 0.5*(Abb_s + Agg_s)*wkhz, A_coeff[i]
        )
    
    eq2 = sym.Eq(
        m*(2*A[a+a]*(A[b+b] - A[g+g]) - 12*(A[a+b]**2 - A[a+g]**2) - A[b+b]**2 + A[g+g]**2)*(q/2) - 0.5*m*(Agg_s - Abb_s)*wkhz, B_coeff[i]
        )
    
    eq3 = sym.Eq(
        -m*(-2*A[a+a]*A[b+g] + 12*A[a+b]*A[a+g] + A[b+g]*(A[b+b] + A[g+g]))*q - m*Abg_s*wkhz, C_coeff[i]
    )
    # Solve the system
    solution = sym.solve([eq1, eq2, eq3], (Abb_s, Agg_s, Abg_s))
    solutions_cs.append(solution)

print(solutions_cs)

#saving solutions
Axx_s = np.mean([solutions_cs[1][Axx_s], solutions_cs[2][Axx_s]])
Ayy_s = np.mean([solutions_cs[0][Ayy_s], solutions_cs[2][Ayy_s]])
Azz_s = np.mean([solutions_cs[0][Azz_s], solutions_cs[1][Azz_s]])

Ayz_s = solutions_cs[0][Ayz_s]
Axz_s = solutions_cs[1][Axz_s]
Axy_s = solutions_cs[2][Axy_s]

print('Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: \n', Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s)


[{Ayy_s: 7.55930884099196e-6, Azz_s: 8.53759572266055e-6, Ayz_s: 5.65763885763099e-6}, {Axx_s: 1.01172260797987e-5, Azz_s: 7.77263806014005e-6, Axz_s: 5.06948541278778e-6}, {Axx_s: 9.01741723790991e-6, Ayy_s: 5.82624838695923e-6, Axy_s: 3.97264053129021e-6}]
Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: 
 9.56732165885429e-6 6.69277861397560e-6 8.15511689140030e-6 5.65763885763099e-6 5.06948541278778e-6 3.97264053129021e-6


In [10]:
#Quadrupolar Tensor in Tenon Frame
Q_T = np.zeros((3,3))
Q_T[0,0] = best_Axx_Q; Q_T[0,1] = best_Axy_Q; Q_T[0,2] = best_Axz_Q;
Q_T[1,0] = best_Axy_Q; Q_T[1,1] = best_Ayy_Q; Q_T[1,2] = best_Ayz_Q;
Q_T[2,0] = best_Axz_Q; Q_T[2,1] = best_Ayz_Q; Q_T[2,2] = best_Azz_Q;



#CSA tensor in tenon frame
CS_T = np.zeros((3,3))
CS_T[0,0] = Axx_s; CS_T[0,1] = Axy_s; CS_T[0,2] = Axz_s;
CS_T[1,0] = Axy_s; CS_T[1,1] = Ayy_s; CS_T[1,2] = Ayz_s;
CS_T[2,0] = Axz_s; CS_T[2,1] = Ayz_s; CS_T[2,2] = Azz_s;

print('Chemical Shift tensor (tenon frame): \n', CS_T, '\n')
print('Quadrupolar tensor (MHz) (tenon frame): \n', Q_T/10**3, '\n')
V = Q_T*(2*Ispin*(2*Ispin - 1))/10**3

Chemical Shift tensor (tenon frame): 
 [[9.56732166e-06 3.97264053e-06 5.06948541e-06]
 [3.97264053e-06 6.69277861e-06 5.65763886e-06]
 [5.06948541e-06 5.65763886e-06 8.15511689e-06]] 

Quadrupolar tensor (MHz) (tenon frame): 
 [[-0.03378649 -0.1806763   0.1825446 ]
 [-0.1806763   0.24031259  0.0495395 ]
 [ 0.1825446   0.0495395  -0.2065261 ]] 



In [11]:
#following the Voseggard et al. paper for principal frame parameters JOURNAL OF MAGNETIC RESONANCE, Series A 122, 111 – 119 ( 1996 ) ARTICLE NO. 0186

#Calculate Quadrupolar Tensor in PAS

sorted_eigenvalues_Q, dc_Q, quad_avg, eigenvalues_Q, eigenvectors_Q = sort_eigenvalues(Q_T*(2*Ispin*(2*Ispin - 1))/10**3)
# print('Sorted Eigenvalues of Quadrupolar diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues_Q, '\n') 

# Vyy = (sorted_eigenvalues_Q[0])*(2*Ispin*(2*Ispin - 1)) 
# Vxx = (sorted_eigenvalues_Q[1])*(2*Ispin*(2*Ispin - 1))
# Vzz = (sorted_eigenvalues_Q[2])*(2*Ispin*(2*Ispin - 1)) 

Vyy = (sorted_eigenvalues_Q[0])
Vxx = (sorted_eigenvalues_Q[1])
Vzz = (sorted_eigenvalues_Q[2]) 

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================ \n')

#Calculate CSA Tensor in PAS

sorted_eigenvalues_csa, dc_csa, csa_avg, eigenvalues_csa, eigenvectors_csa = sort_eigenvalues(CS_T*10**6)
# print('Sorted Eigenvalues of CSA diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues_csa, '\n') 

csyy = -(sorted_eigenvalues_csa[0]) 
csxx = -(sorted_eigenvalues_csa[1])
cszz = -(sorted_eigenvalues_csa[2]) 

print('CSA Tensor Components δyy, δxx, δzz: \n', csyy, csxx, cszz)
print('CSA direction cosine:\n', dc_csa)



 Unsorted Eigenvalues:
 [-2.13803583  0.14028272  1.99775311] 

 Unsorted Eigenvectors:
 [[ 0.5763086   0.66719276 -0.47193456]
 [ 0.23939356  0.41432204  0.87808198]
 [-0.78138283  0.61902429 -0.0790557 ]] 

Sorted Eigenvalues: 
 [ 0.14028272  1.99775311 -2.13803583] 

Sorted Eigenvectors: 
 [[ 0.66719276 -0.47193456  0.5763086 ]
 [ 0.41432204  0.87808198  0.23939356]
 [ 0.61902429 -0.0790557  -0.78138283]] 

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 0.1402827196321036 1.9977531092371428 -2.1380358288692465

 Unsorted Eigenvalues:
 [18.02329882  4.70362553  1.68829281] 

 Unsorted Eigenvectors:
 [[ 0.60562321  0.79121956 -0.08480643]
 [ 0.51517219 -0.47107459 -0.71602119]
 [ 0.60648013 -0.38994914  0.69290802]] 

Sorted Eigenvalues: 
 [ 4.70362553  1.68829281 18.02329882] 

Sorted Eigenvectors: 
 [[ 0.79121956 -0.08480643  0.60562321]
 [-0.47107459 -0.71602119  0.51517219]
 [-0.38994914  0.69290802  0.60648013]] 

CSA Tensor Components δyy, δxx, δzz: 
 -4.7036255339274 -1.68829281

In [12]:
#Quadrupolar tensor parameters
cq = Vzz
etaq = (Vyy - Vxx)/Vzz

#CSA tensor parameters
iso_cs = np.mean([cszz, csyy, csxx]) 
csa = cszz - iso_cs

etas = (csyy - csxx)/csa


table = [['cq (MHz)', cq], ['etaq', etaq ], ['iso_cs (ppm)', iso_cs], ['csa (ppm)', csa], ['etas', etas] ] #converting Hz to ppm (Should be multiplied by 10**6)
print(tabulate(table, headers=['Quantity', 'Fit Value']))

Quantity        Fit Value
------------  -----------
cq (MHz)        -2.13804
etaq             0.868774
iso_cs (ppm)    -8.13841
csa (ppm)       -9.88489
etas             0.305045


**Euler Angles Relating Quad Tensor Tenon--> PAS Frame**

In [13]:
print('\nEFG Direction Cosine\n')
print(dc_Q)

print('\nCS direction cosine\n')
print(dc_csa)

a_Q, b_Q, g_Q = get_euler_angles(dc_Q)
print("\nCalculated Euler angles (degrees):\n")
print('alpha:', a_Q, 'beta:', b_Q, 'gamma:', g_Q,'\n')


EFG Direction Cosine

[[-0.47193456  0.66719276  0.5763086 ]
 [ 0.87808198  0.41432204  0.23939356]
 [-0.0790557   0.61902429 -0.78138283]]

CS direction cosine

[[-0.08480643  0.79121956  0.60562321]
 [-0.71602119 -0.47107459  0.51517219]
 [ 0.69290802 -0.38994914  0.60648013]]

Calculated Euler angles (degrees):

alpha: -82.72214206781811 beta: 141.38736055116075 gamma: -22.55757184571218 



In [14]:
# Eigenvectors from ASICS values
alpha_efg = 82.1
beta_efg = 37
gamma_efg = 335

eigenvectors_efg = Rabc(alpha_efg, beta_efg, gamma_efg)
print('Eigenvectors for EFG tensor from ASICS\n',eigenvectors_efg)

get_euler_angles(eigenvectors_efg)


Eigenvectors for EFG tensor from ASICS
 [[ 0.51809107  0.65885367 -0.54542964]
 [-0.85131644  0.45888179 -0.25433802]
 [ 0.08271619  0.59610348  0.79863551]]


(82.1, 37.0, -24.999999999999986)

**Euler Angles Relating Quad Tensor--> CS Tensor**

In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

# CSA_Q = np.matmul(np.linalg.inv(dc_Q), (dc_csa))
CSA_Q = np.matmul(dc_Q.transpose(), dc_csa)
print('Eigenvectors for CSA --> EFG Frame\n',CSA_Q)

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Eigenvectors for CSA --> EFG Frame
 [[-0.64348056 -0.75621826  0.11860318]
 [ 0.0756813   0.09133138  0.99294054]
 [-0.76171196  0.64791397 -0.00153838]]
Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -40.38456618513448 chi: 90.08814295366408 xi: -83.1884969537895 



In [16]:
# Eigenvectors for CSA --> EFG Frame from ASICS
psi_A = 362.3
chi_A = 87.8
xi_A = 75.2

eigenvectors_csa_Q = Rabc(psi_A, chi_A, xi_A)


psi_A, chi_A, xi_A = get_euler_angles(eigenvectors_csa_Q)

print('Eigenvectors for CSA --> Quadrupolar Frame from ASICS\n',eigenvectors_csa_Q)
print('Euler angles from ASICS\n','psi:', psi_A, 'chi:', chi_A, 'xi:', xi_A, '\n')

Eigenvectors for CSA --> Quadrupolar Frame from ASICS
 [[-0.02900225  0.96643804 -0.25525747]
 [-0.04733583  0.25375051  0.96611076]
 [ 0.99845791  0.04010221  0.03838781]]
Euler angles from ASICS
 psi: 2.2999999999999954 chi: 87.8 xi: 75.2 



In [17]:
# Find Quadrupolar Tensor and rotate in tenon frame
# Values from ASICS
cq = 2.18
etaq = 0.915
a1 = 82.1
b1 = 37
g1 = 335

V_PAS = np.zeros((3,3))
V_PAS[0,0] = -(1 + etaq) * cq/2
V_PAS[1,1] = -(1 - etaq) * cq/2
V_PAS[2,2] = cq

#Transformation between PAS --> Tenon frame
U = Rabc(a1, b1, g1)
V_T = np.matmul(np.matmul(U, V_PAS), np.linalg.inv(U))
print('Quadrupolar tensor in PAS: \n',V_PAS)
print('Quadrupolar tensor in tenon: \n',V_T)

Quadrupolar tensor in PAS: 
 [[-2.08735  0.       0.     ]
 [ 0.      -0.09265  0.     ]
 [ 0.       0.       2.18   ]]
Quadrupolar tensor in tenon: 
 [[ 0.0480345   1.19505122 -1.07544707]
 [ 1.19505122 -1.39127544 -0.32116624]
 [-1.07544707 -0.32116624  1.34324094]]


In [18]:
# Find Rotation Matrix and Rotation angles

# *************** Calculation for CSA ************************
# Quadrupolar Tensor in PAS
Q_PAS = np.zeros((3,3))
Q_PAS[0,0] = Vxx/(2*Ispin*(2*Ispin - 1));
Q_PAS[1,1] = Vyy/(2*Ispin*(2*Ispin - 1));
Q_PAS[2,2] = Vzz/(2*Ispin*(2*Ispin - 1));

#CSA tensor in PAS
CS_PAS = np.zeros((3,3))
CS_PAS[0,0] = -csxx; 
CS_PAS[1,1] = -csyy;
CS_PAS[2,2] = -cszz;
print('Calculation for CSA Tensor: \n')

# Find eigenvalues and eigenvectors of original matrix

eigenvalues, eigenvectors = np.linalg.eig(CS_PAS) 
print('Eigenvalues of CSA (PAS) tensor \n', eigenvalues, '\n')
print('Eigenvectors of CSA (PAS) tensor \n', eigenvectors, '\n')
# Calculate the eigenvalues of the rotated matrix A_rot

eigenvalues_rot, eigenvectors_rot = np.linalg.eig(CS_T)
print('Eigenvalues of CSA (Tenon) tensor \n', eigenvalues_rot, '\n')
print('Eigenvectors of CSA (Tenon) tensor \n', eigenvectors_rot, '\n')

b = np.degrees(np.arccos(CS_T[2,2]))
a = np.degrees(np.arctan(CS_T[2,1]/CS_T[2,0]))
g = np.degrees(np.arctan(-CS_T[1,2]/CS_T[0,2]))
print("Calculated Euler angles (degrees):")
print(a, b, g,'\n')






Calculation for CSA Tensor: 

Eigenvalues of CSA (PAS) tensor 
 [ 1.68829281  4.70362553 18.02329882] 

Eigenvectors of CSA (PAS) tensor 
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]] 

Eigenvalues of CSA (Tenon) tensor 
 [1.80232988e-05 4.70362553e-06 1.68829281e-06] 

Eigenvectors of CSA (Tenon) tensor 
 [[ 0.60562321  0.79121956 -0.08480643]
 [ 0.51517219 -0.47107459 -0.71602119]
 [ 0.60648013 -0.38994914  0.69290802]] 

Calculated Euler angles (degrees):
48.1383064015336 89.9995327462207 -48.1383064015336 

